In [0]:



df_raw= spark.read.format("delta").load("/Volumes/workspace/default/api/landing/")
df_raw.show()




from pyspark.sql.types import *

schema = StructType([
    StructField("name", StringType()),
    StructField("dt", LongType()),
    StructField("main", StructType([
        StructField("temp", DoubleType()),
        StructField("humidity", IntegerType())
    ])),
    StructField("weather", ArrayType(StructType([
        StructField("description", StringType())
    ])))
])

from pyspark.sql.functions import from_json, col

df_parsed = df_raw.withColumn(
    "json_data",
    from_json(col("raw_data"), schema)
)

df_parsed.printSchema()

df_clean = df_parsed.select(
    col("json_data.name").alias("city"),
    col("json_data.main.temp").alias("temperature"),
    col("json_data.main.humidity").alias("humidity"),
    col("json_data.weather")[0]["description"].alias("weather")
)

display(df_clean)

In [0]:
from pyspark.sql.functions import col, from_unixtime, current_timestamp

from pyspark.sql.functions import col, from_unixtime, current_timestamp

from pyspark.sql.functions import from_json, col


df_silver = df_parsed.select(
    col("json_data.name").alias("city"),
    col("json_data.main.temp").alias("temperature"),
    col("json_data.main.humidity").alias("humidity"),
    col("json_data.weather")[0]["description"].alias("weather"),
    col("json_data.dt").alias("timestamp")
)

df_silver = df_silver.withColumn("event_time", from_unixtime("timestamp")) \
                     .withColumn("ingestion_time", current_timestamp())


df_silver = df_silver.dropDuplicates(["city", "timestamp"])


display(df_silver)




In [0]:
df_silver.filter(col("temperature").isNull()).count()
df_silver.filter(col("city").isNull()).count()



dbutils.fs.rm("/Volumes/workspace/default/api/processed/", True)

from pyspark.sql.functions import to_date

df_silver = df_silver.withColumn("date", to_date("event_time"))

df_silver.write \
    .option("mergeSchema", "true") \
    .mode("append") \
    .format("delta") \
    .save("/Volumes/workspace/default/api/processed/")